# 02 — Exploratory Data Analysis

**Phase 1, Week 2** — Profile the public Kaggle Smart Meter Electricity Consumption Dataset: feature distributions, temporal load profiles, weather correlations, and anomaly label baseline.

**Scope:** Analysis and visualization only. No anomaly detection or forecasting models.

## 1. Data Loading

**Why:** Every downstream analysis must start from the validated ingestion pipeline established in Week 1. We reuse `load_smart_meter_data` rather than reading the CSV directly.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Allow imports from src/ when running locally from notebooks/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.ingest_data import find_dataset_csv, get_project_root, load_smart_meter_data
from src.visualization.visualize import (
    add_temporal_features,
    plot_consumption_timeseries,
    plot_correlation_heatmap,
    plot_feature_histograms,
    plot_hourly_load_profile,
    plot_weekly_load_profile,
)

sns.set_theme(style="whitegrid")

csv_path = find_dataset_csv(get_project_root())
df = load_smart_meter_data(csv_path)

print(f"Loaded: {csv_path}")
print(f"Shape: {df.shape}")
print(f"Nulls:  {df.isna().sum().sum()}  (verified 0 in Week 1)")
print(f"Date range: {df['Timestamp'].min()} → {df['Timestamp'].max()}")
print("Week 1 continuity check: PASS — continuous 30-minute series.")

## 2. Feature Distributions

**Why:** Understanding the shape and spread of each numeric feature reveals skewness, outliers, and whether values are uniformly distributed — important context before modeling on normalized data.

In [ ]:
fig = plot_feature_histograms(df)
plt.show()

## 3. Temporal Load Profiles

**Why:** Smart meter data has strong diurnal and weekly patterns. Boxplots by hour and day of week reveal baseline seasonality that anomaly detectors must distinguish from true outliers.

In [ ]:
df = add_temporal_features(df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_hourly_load_profile(df, ax=axes[0])
plot_weekly_load_profile(df, ax=axes[1])
plt.tight_layout()
plt.show()

peak_hour = df.groupby("hour")["Electricity_Consumed"].mean().idxmax()
print(f"Peak mean consumption hour: {peak_hour}:00")

## 4. Weather Correlations

**Why:** Temperature, humidity, and wind can drive heating/cooling load. A correlation matrix quantifies linear relationships before Phase 2 feature selection.

In [ ]:
fig = plot_correlation_heatmap(df)
plt.show()

corr = df[
    ["Electricity_Consumed", "Temperature", "Humidity", "Wind_Speed", "Avg_Past_Consumption"]
].corr()["Electricity_Consumed"].drop("Electricity_Consumed")
print("Correlation with Electricity_Consumed:")
print(corr.sort_values(ascending=False).to_string())

## 5. Anomaly Label Baseline

**Why:** Phase 2 unsupervised methods need a baseline class imbalance reference. The pre-assigned `Anomaly_Label` column (`Normal` / `Abnormal`) provides an evaluation anchor — not used for training in Phase 2.

In [ ]:
label_counts = df["Anomaly_Label"].value_counts()
label_pct = df["Anomaly_Label"].value_counts(normalize=True) * 100

print("Anomaly label counts:")
print(label_counts.to_string())
print("\nPercentages:")
print(label_pct.round(2).astype(str) + "%")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, ax=ax, palette="Set2")
ax.set_title("Anomaly Label Distribution")
ax.set_xlabel("Label")
ax.set_ylabel("Count")
for i, (label, count) in enumerate(label_counts.items()):
    ax.text(i, count + 20, f"{label_pct[label]:.1f}%", ha="center")
plt.tight_layout()
plt.show()

imbalance_ratio = label_counts["Normal"] / label_counts["Abnormal"]
print(f"\nImbalance ratio (Normal : Abnormal): {imbalance_ratio:.1f} : 1")

## 6. Time-Series Overview

**Why:** A full-series plot with a 24-hour rolling average smooths 30-minute noise and reveals longer-term trends or regime shifts across the ~104-day window.

In [ ]:
fig = plot_consumption_timeseries(df, window=48)
plt.show()

---

**Phase 1 Week 2 EDA complete.** Findings are documented in `docs/eda-insights.md`. Do not proceed to anomaly detection (Phase 2) in this notebook.